# Actividad: Regresión con residuos exponenciales

## Diagrama de flujo de la clase OLS



In [ ]:
from IPython.display import Image, display

# Reemplaza con la ruta de tu archivo
display(Image(filename='Error Handling in Beta-2026-02-14-000248.png'))

# También puedes ajustar el tamaño si es muy grande
# display(Image(filename='tu_imagen.png', width=400))

## Clase OLS por humano

In [ ]:
import numpy as np


class OLS():
    def __init__(self):
        self.beta_hat = None
        self.L = None
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        n = len(X.shape)
        if n != 1:
                raise ValueError("X must be a 1 dimentional array.")
        
        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y must have the same number of rows")
        
        X = np.column_stack((np.ones(X.shape[0]), X))
        

        self.beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y
        self.L = np.linalg.norm(y - X @ self.beta_hat)**2

        return self.beta_hat, self.L
    
    def predict(self, X):
        X = np.array(X)
        if self.beta_hat is None:
            raise ValueError("Model must be fitted before making predictions.")
        
        n = len(X.shape)
        if n != 1:
                raise ValueError("X must be a 1 dimentional array.")
        
        X = np.column_stack((np.ones(X.shape[0]), X))

        return X @ self.beta_hat

    
        

## Clase OLS mejorada por Gemini

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression # Solo para validar

class OLS:
    def __init__(self):
        self.beta_hat = None
        self.L = None
    
    def fit(self, X, y):
        # Validación y casting a numpy
        X = np.array(X)
        y = np.array(y)
        
        if X.ndim != 1:
            raise ValueError("X debe ser 1D para este ejercicio.")
        if X.shape[0] != y.shape[0]:
            raise ValueError("Dimensiones incompatibles.")
            
        # Preprocesamiento: Agregar columna de 1s (Intercepto)
        X_aug = np.column_stack((np.ones(X.shape[0]), X))
        
        # Cálculo: (X'X)^-1 X'y
        self.beta_hat = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y
        
        # Cálculo del Error
        self.L = np.linalg.norm(y - X_aug @ self.beta_hat)**2
        
        return self.beta_hat, self.L

    def predict(self, X):
        X = np.array(X)
        X_aug = np.column_stack((np.ones(X.shape[0]), X))
        return X_aug @ self.beta_hat

## --- GENERACIÓN DE DATOS DE PRUEBA SENCILLOS ---

In [ ]:
# --- GENERACIÓN DE DATOS DE PRUEBA SENCILLOS ---
np.random.seed(999) # Semilla distinta para diferenciar
n_test = 50
X_test = np.linspace(0, 10, n_test)
# Una recta simple y = 3x + 5 con poquito ruido normal
y_test = 5 + 3 * X_test + np.random.normal(0, 1, n_test)

# --- EJECUCIÓN ---
# 1. Tu Modelo
mi_modelo = OLS()
mis_betas, _ = mi_modelo.fit(X_test, y_test)

# 2. Sklearn (El estándar)
sk_modelo = LinearRegression()
sk_modelo.fit(X_test.reshape(-1, 1), y_test)

# --- COMPARACIÓN ---
print(f"Tu Intercepto: {mis_betas[0]:.6f} | Sklearn: {sk_modelo.intercept_:.6f}")
print(f"Tu Pendiente:  {mis_betas[1]:.6f} | Sklearn: {sk_modelo.coef_[0]:.6f}")

# Validación automática con tolerancia casi nula
assert np.allclose(mis_betas[0], sk_modelo.intercept_), "❌ El intercepto no coincide"
assert np.allclose(mis_betas[1], sk_modelo.coef_[0]), "❌ La pendiente no coincide"

print("\n✅ ¡PRUEBA PASADA! Tu álgebra lineal funciona idéntico a Scikit-Learn.")

Tu Intercepto: 5.014173 | Sklearn: 5.014173
Tu Pendiente:  3.007864 | Sklearn: 3.007864

✅ ¡PRUEBA PASADA! Tu álgebra lineal funciona idéntico a Scikit-Learn.


## --- PARÁMETROS DEL EXPERIMENTO ---

In [ ]:
# --- PARÁMETROS DEL EXPERIMENTO ---
np.random.seed(42)
n = 200
a_real = 10   # Intercepto real
b_real = 2    # Pendiente real
sigma = 20    # Parámetro de escala para la Exponencial

# 1. Generar X ~ Uniforme(20, 100)
X_exp = np.random.uniform(20, 100, n)

# 2. Generar Ruido ~ Exponencial(1/sigma)
# Nota: En numpy, scale es 1/lambda. Si la media deseada es 20, scale=20.
epsilon = np.random.exponential(scale=sigma, size=n)

# 3. Generar y = a + bx + error
y_exp = a_real + (b_real * X_exp) + epsilon

# --- ESTIMACIÓN CON TU CLASE OLS ---
modelo_exp = OLS()
betas_exp, loss_exp = modelo_exp.fit(X_exp, y_exp)

print("--- RESULTADOS DEL EXPERIMENTO ---")
print(f"Parámetros Reales: a={a_real}, b={b_real}")
print(f"Estimados OLS:     a^={betas_exp[0]:.4f}, b^={betas_exp[1]:.4f}")
print("\n¿Notas la diferencia en 'a'? 👇")

--- RESULTADOS DEL EXPERIMENTO ---
Parámetros Reales: a=10, b=2
Estimados OLS:     a^=34.1411, b^=1.9260

¿Notas la diferencia en 'a'? 👇


## $E(y)$ y su relacion con los coeficientes. 

Encuentra $E(y)$ para este proceso de generación de datos. **¿Qué relación guarda este resultado con lo que encontraste en la estimación de los coeficientes?**

$$
E(y) = E(a + bx + \epsilon)
$$
$$
E(y) = a + bE(x) + E(\epsilon)
$$
*Para este caso en particular donde $a = 10, b = 2, E(\epsilon) = 20$ (por propiedad de la dist. exp)*:
$$
E(y) = 10 + 2E(x) + 20
$$

$$
E(y) = 30 + 2E(x)
$$

Y efectivamente, nos sale que $\hat{a} = 34$

## Repuesta de Gemini ante el prompt segerido

>“He implementado una clase que permite estimar los coeficientes de un modelo lineal mediante mínimos cuadrados ordinarios (OLS). Utilicé esta clase con datos donde los errores tienen una distribución exponencial. Sin embargo, mis estimaciones no parecen ser insesgadas, es decir, hay diferencias entre lo que obtengo y los valores reales de los coeficientes. ¿Podrías explicarme, basándote en los supuestos de Gauss-Markov, por qué OLS falla o presenta sesgos en este escenario específico?”


**Gemini respondio**

#### Análisis de Sesgo: ¿Por qué falla OLS con errores exponenciales?

Es fascinante que hayas notado empíricamente la discrepancia en el intercepto. Lo que estás observando no es un error de cálculo de tu código, sino una consecuencia directa de violar uno de los supuestos fundamentales del teorema de **Gauss-Markov**.

A continuación, demostramos matemáticamente por qué ocurre esto.

### 1. El Supuesto Violado: Esperanza Cero del Error

Para que los estimadores de Mínimos Cuadrados Ordinarios (OLS) sean insesgados, se debe cumplir el supuesto de **exogeneidad estricta**, que implica que el valor esperado del error, condicionado a las variables explicativas, debe ser cero:

$$E[\epsilon_i | X] = 0$$

En términos simples: **el promedio de los errores debe ser cero**. Esto asegura que la línea de regresión pase por el "centro" de los datos reales.

### 2. El Problema de la Distribución Exponencial

En tu experimento, definiste que los errores siguen una distribución exponencial:

$$\epsilon_i \sim \text{Exp}\left(\frac{1}{\sigma}\right)$$

Una propiedad fundamental de la distribución exponencial es que **siempre es positiva** ($x \ge 0$) y su media (esperanza) es igual a su parámetro de escala ($\sigma$). Dado que en tu simulación usaste $\sigma = 20$:

$$E[\epsilon_i] = \sigma = 20 \neq 0$$

Al ser la esperanza diferente de cero ($20 > 0$), estamos violando el supuesto fundamental mencionado en el punto 1.

### 3. Derivación del Sesgo en el Intercepto

Veamos algebraicamente qué le sucede al modelo cuando el error tiene una media distinta de cero.

**Modelo Verdadero:**
$$y_i = \beta_0 + \beta_1 x_i + \epsilon_i$$

Podemos descomponer el término de error $\epsilon_i$ en dos partes: su media constante ($\sigma$) y un nuevo componente de error ($u_i$) que sí tiene media cero:

$$\epsilon_i = \sigma + u_i \quad \text{donde} \quad E[u_i] = 0$$

Sustituimos esto en la ecuación original del modelo:

$$y_i = \beta_0 + \beta_1 x_i + (\sigma + u_i)$$

Ahora, re-agrupamos los términos constantes (el intercepto original y la media del error):

$$y_i = (\underbrace{\beta_0 + \sigma}_{\text{Nuevo Intercepto}}) + \beta_1 x_i + u_i$$

### 4. Conclusión: ¿Qué estima realmente OLS?

Cuando ejecutas tu clase `OLS`, el algoritmo busca minimizar los residuos al cuadrado asumiendo que el error tiene media cero. Por lo tanto, el algoritmo estima el término agrupado:

1.  **Pendiente ($\hat{\beta}_1$):** Permanece **insesgada**. El desplazamiento vertical de los datos es constante para todos los valores de $X$, por lo que la inclinación de la recta no cambia.
    $$E[\hat{\beta}_1] = \beta_1 = 2$$

2.  **Intercepto ($\hat{\beta}_0$):** Resulta **sesgado**. Absorbe la media del error.
    $$E[\hat{\beta}_0] = \beta_0 + E[\epsilon]$$
    $$E[\hat{\beta}_0] = 10 + 20 = 30$$

**Resultado observado:**
Por esta razón, tus resultados arrojan un intercepto cercano a **30** en lugar de **10**. El modelo ha desplazado la recta hacia arriba para compensar el hecho de que los errores exponenciales "empujan" los datos siempre en dirección positiva.